In [1]:
import math
import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerAE(nn.Module):

    def __init__(
        self,
        input_dim,
        d_model=64,
        nhead=4,
        num_layers=2,
        ff_dim=128,
        dropout=0.1,
    ):
        super().__init__()

        self.input_proj = nn.Linear(
            input_dim,
            d_model,
        )

        self.positional = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        self.output_proj = nn.Linear(
            d_model,
            input_dim,
        )

    def forward(self, x):

        x = self.input_proj(x)

        x = self.positional(x)

        x = self.encoder(x)

        x = self.output_proj(x)

        return x

### Data


----------> TRAIN ON NORMAL DATA!!!!!!!!!

Yes. For your TransformerAE, the key idea is:

One sample = one sequence of observations.

Your model expects input with shape:

(batch_size, sequence_length, input_dim)


```
X_train
│
├── 1000 sequences
│
├── each sequence
│   ├── 10 timesteps
│   │
│   └── each timestep
│       └── 3 features
│
└── shape = (1000, 10, 3)
```

In [5]:
import torch

n_train = 1000
n_test = 200

seq_len = 10
input_dim = 3

X_train = torch.randn(n_train, seq_len, input_dim)
X_test = torch.randn(n_test, seq_len, input_dim)

print(X_train.shape)
print(X_test.shape)

#https://chatgpt.com/c/6a6f67db-cc90-83e9-a2d3-b867db1aa44e
# servers data

torch.Size([1000, 10, 3])
torch.Size([200, 10, 3])


## Training

In [7]:
X_val = X_test


from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(
    TensorDataset(X_train),
    batch_size=32,
    shuffle=True,
)

val_loader = DataLoader(
    TensorDataset(X_val),
    batch_size=32,
    shuffle=False,
)

In [8]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerAE(
    input_dim=3,
    d_model=64,
    nhead=4,
    num_layers=2,
    ff_dim=128,
    dropout=0.1,
).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

epochs = 10



for epoch in range(epochs):

    # =========================
    # TRAIN
    # =========================
    model.train()

    train_loss = 0.0

    for (x,) in train_loader:
        x = x.to(device)

        optimizer.zero_grad()

        x_hat = model(x)

        loss = criterion(x_hat, x)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)

    # =========================
    # VALIDATION
    # =========================
    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for (x,) in val_loader:
            x = x.to(device)

            x_hat = model(x)

            loss = criterion(x_hat, x)

            val_loss += loss.item() * x.size(0)

    val_loss /= len(val_loader.dataset)

    print(
        f"Epoch {epoch + 1:02d}/{epochs} "
        f"| train loss: {train_loss:.6f} "
        f"| val loss: {val_loss:.6f}"
    )

Epoch 01/10 | train loss: 0.260310 | val loss: 0.035063
Epoch 02/10 | train loss: 0.040496 | val loss: 0.015039
Epoch 03/10 | train loss: 0.024098 | val loss: 0.006693
Epoch 04/10 | train loss: 0.016572 | val loss: 0.003780
Epoch 05/10 | train loss: 0.012477 | val loss: 0.002928
Epoch 06/10 | train loss: 0.010203 | val loss: 0.002337
Epoch 07/10 | train loss: 0.008464 | val loss: 0.001806
Epoch 08/10 | train loss: 0.007486 | val loss: 0.001553
Epoch 09/10 | train loss: 0.006582 | val loss: 0.001418
Epoch 10/10 | train loss: 0.005798 | val loss: 0.001456


Training loss: one scalar used for backward().
Anomaly score: one value per sequence, used to detect anomalies.

In [9]:
# adding reconstruct error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = TransformerAE(
    input_dim=3,
    d_model=64,
    nhead=4,
    num_layers=2,
    ff_dim=128,
    dropout=0.1,
).to(device)


criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)


train_loader = DataLoader(
    TensorDataset(X_train),
    batch_size=32,
    shuffle=True,
)

val_loader = DataLoader(
    TensorDataset(X_val),
    batch_size=32,
    shuffle=False,
)


def reconstruction_scores(x, x_hat):
    """
    Return one reconstruction/anomaly score per sequence.

    x, x_hat:
        (batch, sequence_length, input_dim)

    returns:
        (batch,)
    """

    error = (x_hat - x) ** 2

    scores = error.mean(dim=(1, 2))

    return scores


epochs = 20

for epoch in range(epochs):

    # ==================================================
    # TRAIN
    # ==================================================

    model.train()

    train_loss = 0.0

    for (x,) in train_loader:

        x = x.to(device)

        optimizer.zero_grad()

        x_hat = model(x)

        # One scalar for the batch
        loss = criterion(x_hat, x)

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)


    # ==================================================
    # VALIDATION
    # ==================================================

    model.eval()

    val_loss = 0.0
    val_scores = []

    with torch.no_grad():

        for (x,) in val_loader:

            x = x.to(device)

            x_hat = model(x)

            # Scalar validation loss
            loss = criterion(x_hat, x)

            val_loss += loss.item() * x.size(0)

            # One score per sequence
            scores = reconstruction_scores(
                x,
                x_hat,
            )

            val_scores.append(
                scores.cpu()
            )

    val_loss /= len(val_loader.dataset)

    val_scores = torch.cat(val_scores)

    print(
        f"Epoch {epoch + 1:02d}/{epochs} "
        f"| train loss: {train_loss:.6f} "
        f"| val loss: {val_loss:.6f} "
        f"| val score mean: {val_scores.mean():.6f}"
    )

Epoch 01/20 | train loss: 0.199049 | val loss: 0.031826 | val score mean: 0.031826
Epoch 02/20 | train loss: 0.033180 | val loss: 0.009298 | val score mean: 0.009298
Epoch 03/20 | train loss: 0.017712 | val loss: 0.004232 | val score mean: 0.004232
Epoch 04/20 | train loss: 0.012437 | val loss: 0.003003 | val score mean: 0.003003
Epoch 05/20 | train loss: 0.009650 | val loss: 0.002326 | val score mean: 0.002326
Epoch 06/20 | train loss: 0.007869 | val loss: 0.001939 | val score mean: 0.001939
Epoch 07/20 | train loss: 0.006710 | val loss: 0.001577 | val score mean: 0.001577
Epoch 08/20 | train loss: 0.005880 | val loss: 0.001546 | val score mean: 0.001546
Epoch 09/20 | train loss: 0.005131 | val loss: 0.001177 | val score mean: 0.001177
Epoch 10/20 | train loss: 0.004674 | val loss: 0.001140 | val score mean: 0.001140
Epoch 11/20 | train loss: 0.004137 | val loss: 0.001010 | val score mean: 0.001010
Epoch 12/20 | train loss: 0.003788 | val loss: 0.000962 | val score mean: 0.000962
Epoc

In [12]:
val_scores.shape, val_scores

(torch.Size([200]),
 tensor([0.0002, 0.0003, 0.0003, 0.0003, 0.0005, 0.0001, 0.0002, 0.0019, 0.0003,
         0.0002, 0.0004, 0.0003, 0.0004, 0.0003, 0.0024, 0.0002, 0.0004, 0.0009,
         0.0003, 0.0003, 0.0004, 0.0004, 0.0002, 0.0006, 0.0011, 0.0002, 0.0001,
         0.0003, 0.0004, 0.0003, 0.0003, 0.0079, 0.0003, 0.0002, 0.0005, 0.0003,
         0.0003, 0.0006, 0.0003, 0.0004, 0.0004, 0.0001, 0.0002, 0.0003, 0.0010,
         0.0003, 0.0002, 0.0003, 0.0008, 0.0004, 0.0002, 0.0002, 0.0002, 0.0001,
         0.0003, 0.0002, 0.0093, 0.0006, 0.0005, 0.0003, 0.0008, 0.0002, 0.0004,
         0.0003, 0.0005, 0.0007, 0.0003, 0.0003, 0.0003, 0.0005, 0.0003, 0.0003,
         0.0003, 0.0002, 0.0002, 0.0003, 0.0003, 0.0004, 0.0027, 0.0003, 0.0002,
         0.0004, 0.0002, 0.0004, 0.0002, 0.0043, 0.0003, 0.0003, 0.0002, 0.0003,
         0.0003, 0.0003, 0.0003, 0.0003, 0.0002, 0.0005, 0.0004, 0.0003, 0.0009,
         0.0002, 0.0004, 0.0003, 0.0003, 0.0003, 0.0002, 0.0005, 0.0004, 0.0003,
        

## Evaluation

In [13]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score


# ============================================================
# Fake test data
# ============================================================

n_test = 200
seq_len = 10
input_dim = 3

# Normal data
X_test = torch.randn(
    n_test,
    seq_len,
    input_dim,
)

# Ground-truth labels
# 0 = normal
# 1 = anomaly
y_test = np.zeros(n_test, dtype=int)

# Make the last 40 windows anomalous
n_anomalies = 40

y_test[-n_anomalies:] = 1

# Inject a strong abnormal pattern
X_test[-n_anomalies:] += 4.0


# ============================================================
# DataLoader
# ============================================================

from torch.utils.data import DataLoader, TensorDataset

test_loader = DataLoader(
    TensorDataset(X_test),
    batch_size=32,
    shuffle=False,
)


# ============================================================
# Reconstruction / anomaly scores
# ============================================================

model.eval()

test_scores = []

with torch.no_grad():

    for (x,) in test_loader:

        x = x.to(device)

        # Reconstruction
        x_hat = model(x)

        # Squared reconstruction error
        error = (x_hat - x) ** 2

        # One score per sequence
        scores = error.mean(dim=(1, 2))

        test_scores.append(
            scores.cpu()
        )


# Combine batches
test_scores = torch.cat(test_scores).numpy()


# ============================================================
# ROC-AUC
# ============================================================

roc_auc = roc_auc_score(
    y_test,
    test_scores,
)

print(f"ROC-AUC: {roc_auc:.4f}")

ROC-AUC: 1.0000


In [15]:
test_scores.shape

(200,)

In [16]:
# RECOMENDDE when anomalies are rare
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(
    y_test,
    test_scores,
)

print(f"PR-AUC: {pr_auc:.4f}")

PR-AUC: 1.0000


In [17]:
threshold = 0.05 # selected with vak
y_pred = (test_scores >= threshold).astype(int)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)

precision = precision_score(
    y_test,
    y_pred,
)

recall = recall_score(
    y_test,
    y_pred,
)

f1 = f1_score(
    y_test,
    y_pred,
)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

Precision: 1.0000
Recall:    1.0000
F1:        1.0000


In [18]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred,
)

print(cm)

[[160   0]
 [  0  40]]


                    Why
------------------------------------------------
ROC-AUC             overall ranking
PR-AUC              ranking under class imbalance
Recall              how many anomalies detected
Precision           how many alerts are real
F1                  precision/recall tradeoff

## Inference

In [19]:
import torch


def predict_anomaly(
    model,
    X_window,
    device,
):
    """
    X_window:
        numpy array or tensor with shape (seq_len, input_dim)

    Returns:
        anomaly_score: float
    """

    model.eval()

    # Convert to tensor if necessary
    if not isinstance(X_window, torch.Tensor):
        X_window = torch.tensor(
            X_window,
            dtype=torch.float32,
        )

    # Add batch dimension
    X_window = X_window.unsqueeze(0)

    X_window = X_window.to(device)

    with torch.no_grad():

        # Reconstruction
        X_hat = model(X_window)

        # Per-window reconstruction error
        error = (X_hat - X_window) ** 2

        anomaly_score = error.mean().item()

    return anomaly_score

In [20]:
import numpy as np

new_window = np.random.randn(
    10,
    3,
)

score = predict_anomaly(
    model,
    new_window,
    device,
)

print(f"Anomaly score: {score:.6f}")

Anomaly score: 0.000269


In [21]:
threshold = 0.05

is_anomaly = score >= threshold

print("Anomaly" if is_anomaly else "Normal")

Normal
